# DataCoolie Fabric — SQL Database Metadata Sample

Demonstrates using a **Fabric SQL Database** (`datacoolie`) as the DataCoolie
metadata backend (`DatabaseProvider`) instead of a flat JSON file.

## Two phases

| Phase | Controlled by | What it does |
|---|---|---|
| **Setup** | `RUN_SETUP = True` | Applies DDL schema + seeds connections / dataflows / schema hints from `fabric_use_cases.json` |
| **Run** | always executes | Reads metadata from the SQL Database and runs all DataCoolie dataflows via `DatabaseProvider` |

Set `RUN_SETUP = True` on the first run, then flip it to `False` for subsequent runs.  
Set `TRUNCATE_WS = True` together with `RUN_SETUP = True` to re-seed from scratch.

## Prerequisites

1. Attach a Lakehouse to this notebook (provides `/lakehouse/default/` mount).
2. Upload to `Files/metadata/` in that Lakehouse:
   - `fabric_use_cases.json`
   - `schema_mssql.sql` (from `usecase-sim/metadata/database/`)
3. Upload the DataCoolie wheel to `Files/libraries/`.
4. Provision a Fabric SQL Database named `datacoolie` in your workspace.
5. Note the TDS endpoint: `<server>.database.fabric.microsoft.com,1433`.
6. Fill in `SQL_SERVER` and `WORKSPACE_ID` in the **Config** cell below.

In [ ]:
%pip install "/lakehouse/default/Files/libraries/datacoolie-0.1.0-py3-none-any.whl" --force-reinstall
# %pip install datacoolie  # in the future

In [ ]:
notebookutils.session.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Config — update these values before running
# ---------------------------------------------------------------------------

# ABFSS root for the attached Lakehouse (used for log output).
ROOT_PATH = "abfss://DataCoolie_DEV_WS@onelake.dfs.fabric.microsoft.com/lh_bronze.Lakehouse"

# Logs written through FabricPlatform under this Files folder.
BASE_LOG_PATH = f"{ROOT_PATH}/Files/logs/datacoolie"

# Stage filter — empty string runs every stage.
STAGE = ""

# Workspace ID used as the partition key inside the metadata DB.
# All seeded rows and all DatabaseProvider queries are scoped to this value.
WORKSPACE_ID = "fabric-workspace"  # replace with your actual workspace UUID

# Fabric SQL Database TDS endpoint.  Find it in the Fabric portal under
# your SQL Database item ▸ Settings ▸ Connection strings.
SQL_SERVER   = "<server>"  # REPLACE
SQL_DATABASE = "<datacoolie>"   # REPLACE

# SQL auth credentials — leave both empty to use Entra token auth (recommended).
SQL_USER     = ""
SQL_PASSWORD = ""

# Local lakehouse mount paths — usually no need to change these.
LAKEHOUSE_FILES_DIR = "/lakehouse/default/Files"
METADATA_JSON_PATH  = f"{LAKEHOUSE_FILES_DIR}/metadata/fabric_use_cases.json"
SCHEMA_SQL_PATH     = f"{LAKEHOUSE_FILES_DIR}/metadata/schema_mssql.sql"

# ---------------------------------------------------------------------------
# Setup flags
# ---------------------------------------------------------------------------
# Set True on the first run to apply DDL + seed metadata into the SQL Database.
# Flip to False for all subsequent runs.
RUN_SETUP   = True

# Set True (together with RUN_SETUP=True) to delete existing rows for
# WORKSPACE_ID before re-seeding.  Safe to run multiple times.
TRUNCATE_WS = True

In [ ]:
# ---------------------------------------------------------------------------
# Build SQLAlchemy engine for the Fabric SQL Database
# ---------------------------------------------------------------------------
import struct

from sqlalchemy import create_engine

if SQL_USER and SQL_PASSWORD:
    # -----------------------------------------------------------------------
    # Option A: SQL authentication
    # -----------------------------------------------------------------------
    import urllib.parse
    _params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 18 for SQL Server}}"
        f";SERVER={SQL_SERVER}"
        f";DATABASE={SQL_DATABASE}"
        f";UID={SQL_USER}"
        f";PWD={SQL_PASSWORD}"
        f";Encrypt=yes"
        f";TrustServerCertificate=no"
    )
    _conn_str = f"mssql+pyodbc:///?odbc_connect={_params}"
    _meta_engine = create_engine(_conn_str, fast_executemany=True)
else:
    # -----------------------------------------------------------------------
    # Option B: Entra ID token authentication (recommended for Fabric)
    # Acquires a token scoped to Azure SQL / Fabric SQL Database.
    # -----------------------------------------------------------------------
    _raw_token = notebookutils.credentials.getToken("https://database.windows.net/")
    # pyodbc requires the token encoded as UTF-16-LE bytes with a length prefix.
    _token_bytes = _raw_token.encode("UTF-16-LE")
    _token_struct = struct.pack(f"<I{len(_token_bytes)}s", len(_token_bytes), _token_bytes)

    import urllib.parse
    _params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 18 for SQL Server}}"
        f";SERVER={SQL_SERVER}"
        f";DATABASE={SQL_DATABASE}"
        f";Encrypt=yes"
        f";TrustServerCertificate=no"
    )
    _conn_str = f"mssql+pyodbc:///?odbc_connect={_params}"
    # SQL_COPT_SS_ACCESS_TOKEN = 1256
    _meta_engine = create_engine(
        _conn_str,
        fast_executemany=True,
        connect_args={"attrs_before": {1256: _token_struct}},
    )

# Quick connectivity check
with _meta_engine.connect() as _chk:
    print("Connected to Fabric SQL Database:", SQL_DATABASE)

In [ ]:
# ---------------------------------------------------------------------------
# SETUP Phase 1 — Apply DDL schema
# Reads schema_mssql.sql from the lakehouse Files mount and executes each
# statement.  The IF NOT EXISTS guards make this idempotent.
# ---------------------------------------------------------------------------
if RUN_SETUP:
    import re
    from pathlib import Path
    from sqlalchemy import text

    _schema_sql = Path(SCHEMA_SQL_PATH).read_text(encoding="utf-8")
    # Strip line comments before splitting on ';'
    _schema_sql = re.sub(r"--[^\n]*", "", _schema_sql)

    with _meta_engine.begin() as _conn:
        for _stmt in _schema_sql.split(";"):
            _stmt = _stmt.strip()
            if _stmt:
                _conn.execute(text(_stmt))

    print("Schema applied successfully.")

In [ ]:
# ---------------------------------------------------------------------------
# SETUP Phase 2 — Seed metadata from fabric_use_cases.json
# Reads the JSON from the lakehouse Files mount and inserts connections,
# dataflows, and schema hints into the Fabric SQL Database.
# Existing rows with the same primary key are silently skipped (INSERT guard).
# ---------------------------------------------------------------------------
if RUN_SETUP:
    import json
    import uuid
    from datetime import datetime, timezone
    from pathlib import Path

    from sqlalchemy import text
    from sqlalchemy.exc import IntegrityError

    # ------------------------------------------------------------------
    # Inline helpers (mirrors setup_metadata.py, no import needed)
    # ------------------------------------------------------------------
    _UUID_NS = uuid.UUID("da7ac001-e000-4000-8000-000000000000")

    def _name_to_uuid(name: str) -> str:
        return str(uuid.uuid5(_UUID_NS, name))

    def _json_str(obj):
        if obj is None:
            return None
        if isinstance(obj, str):
            return obj
        return json.dumps(obj)

    def _insert_or_skip(conn, sql: str, params: dict) -> None:
        try:
            conn.execute(text(sql), params)
        except IntegrityError:
            pass  # row already exists — skip

    _NOW = datetime.now(timezone.utc).replace(tzinfo=None)

    # ------------------------------------------------------------------
    # SQL templates (MSSQL dialect — reserved words in [brackets])
    # ------------------------------------------------------------------
    _CONN_SQL = """
    INSERT INTO dc_framework_connections
        (connection_id, workspace_id, name, description,
         connection_type, [format], catalog, [database],
         configure, secrets_ref, is_active, version,
         created_at, updated_at, created_by, updated_by)
    VALUES
        (:cid, :ws, :name, :desc,
         :ctype, :fmt, :catalog, :db,
         :configure, :secrets_ref, :active, 1,
         :now, :now, 'notebook_setup', 'notebook_setup')
    """

    _DF_SQL = """
    INSERT INTO dc_framework_dataflows
        (dataflow_id, workspace_id, name, description,
         stage, group_number, execution_order, processing_mode,
         source_connection_id, source_schema, source_table,
         source_query, source_python_function,
         source_watermark_columns, source_configure,
         transform,
         destination_connection_id, destination_schema, destination_table,
         destination_load_type, destination_merge_keys, destination_configure,
         configure, is_active, version,
         created_at, updated_at, created_by, updated_by)
    VALUES
        (:did, :ws, :name, :desc,
         :stage, :gnum, :eorder, :pmode,
         :src_cid, :src_schema, :src_table,
         :src_query, :src_pyfunc,
         :src_wm, :src_conf,
         :transform,
         :dst_cid, :dst_schema, :dst_table,
         :dst_load, :dst_merge, :dst_conf,
         :conf, :active, 1,
         :now, :now, 'notebook_setup', 'notebook_setup')
    """

    _HINT_SQL = """
    INSERT INTO dc_framework_schema_hints
        (schema_hint_id, connection_id, dataflow_id,
         schema_name, table_name, column_name,
         data_type, [format], [precision], scale,
         default_value, ordinal_position, is_active,
         created_at, updated_at)
    VALUES
        (:hid, :cid, :dfid,
         :sname, :tname, :cname,
         :dtype, :fmt, :prec, :scale,
         :defval, :ordinal, :active,
         :now, :now)
    """

    # ------------------------------------------------------------------
    # Load source JSON
    # ------------------------------------------------------------------
    _meta = json.loads(Path(METADATA_JSON_PATH).read_text(encoding="utf-8"))

    # ------------------------------------------------------------------
    # Optional truncate (clears existing rows for WORKSPACE_ID)
    # ------------------------------------------------------------------
    if TRUNCATE_WS:
        with _meta_engine.begin() as _conn:
            _conn.execute(text(
                "DELETE FROM dc_framework_schema_hints "
                "WHERE dataflow_id IN "
                "  (SELECT dataflow_id FROM dc_framework_dataflows WHERE workspace_id = :ws) "
                "OR connection_id IN "
                "  (SELECT connection_id FROM dc_framework_connections WHERE workspace_id = :ws)"
            ), {"ws": WORKSPACE_ID})
            _conn.execute(text(
                "DELETE FROM dc_framework_dataflows WHERE workspace_id = :ws"
            ), {"ws": WORKSPACE_ID})
            _conn.execute(text(
                "DELETE FROM dc_framework_connections WHERE workspace_id = :ws"
            ), {"ws": WORKSPACE_ID})
        print(f"Truncated existing rows for workspace_id={WORKSPACE_ID!r}")

    # ------------------------------------------------------------------
    # Insert connections
    # ------------------------------------------------------------------
    _connections = _meta.get("connections", [])
    with _meta_engine.begin() as _conn:
        for _c in _connections:
            _cid = _c.get("connection_id") or _name_to_uuid(_c["name"])
            _insert_or_skip(_conn, _CONN_SQL, {
                "cid": _cid, "ws": WORKSPACE_ID,
                "name": _c["name"], "desc": _c.get("description"),
                "ctype": _c["connection_type"],
                "fmt": _c.get("format", ""),
                "catalog": _c.get("catalog"), "db": _c.get("database"),
                "configure": _json_str(_c.get("configure")),
                "secrets_ref": _json_str(_c.get("secrets_ref")),
                "active": _c.get("is_active", True),
                "now": _NOW,
            })
    print(f"Seeded {len(_connections)} connection(s).")

    # ------------------------------------------------------------------
    # Insert dataflows
    # ------------------------------------------------------------------
    _dataflows = _meta.get("dataflows", [])
    with _meta_engine.begin() as _conn:
        for _d in _dataflows:
            _did  = _d.get("dataflow_id") or _name_to_uuid(_d["name"])
            _src  = _d.get("source", {}) or {}
            _dst  = _d.get("destination", {}) or {}
            _src_cid = (
                _d.get("source_connection_id")
                or _name_to_uuid(_src.get("connection_name", ""))
            )
            _dst_cid = (
                _d.get("destination_connection_id")
                or _name_to_uuid(_dst.get("connection_name", ""))
            )
            # Merge partition_columns into destination_configure
            _dst_conf = _dst.get("configure") or {}
            if isinstance(_dst_conf, str):
                try:
                    _dst_conf = json.loads(_dst_conf)
                except (ValueError, TypeError):
                    _dst_conf = {}
            if not isinstance(_dst_conf, dict):
                _dst_conf = {}
            _dst_parts = _dst.get("partition_columns")
            if _dst_parts and "partition_columns" not in _dst_conf:
                _dst_conf = dict(_dst_conf)
                _dst_conf["partition_columns"] = _dst_parts

            _insert_or_skip(_conn, _DF_SQL, {
                "did": _did, "ws": WORKSPACE_ID,
                "name": _d["name"], "desc": _d.get("description"),
                "stage": _d.get("stage"),
                "gnum": _d.get("group_number"),
                "eorder": _d.get("execution_order"),
                "pmode": _d.get("processing_mode"),
                "src_cid": _src_cid,
                "src_schema": _src.get("schema_name") or _d.get("source_schema"),
                "src_table":  _src.get("table")  or _d.get("source_table"),
                "src_query":  _src.get("query")  or _d.get("source_query"),
                "src_pyfunc": _src.get("python_function") or _d.get("source_python_function"),
                "src_wm":   _json_str(_src.get("watermark_columns") or _d.get("source_watermark_columns")),
                "src_conf": _json_str(_src.get("configure") or _d.get("source_configure")),
                "transform": _json_str(_d.get("transform")),
                "dst_cid": _dst_cid,
                "dst_schema": _dst.get("schema_name") or _d.get("destination_schema"),
                "dst_table":  _dst.get("table")  or _d.get("destination_table"),
                "dst_load":   _dst.get("load_type") or _d.get("destination_load_type"),
                "dst_merge":  _json_str(_dst.get("merge_keys") or _d.get("destination_merge_keys")),
                "dst_conf":   _json_str(_dst_conf or None),
                "conf": _json_str(_d.get("configure")),
                "active": _d.get("is_active", True),
                "now": _NOW,
            })
    print(f"Seeded {len(_dataflows)} dataflow(s).")

    # ------------------------------------------------------------------
    # Insert schema hints
    # ------------------------------------------------------------------
    _hint_groups = _meta.get("schema_hints", [])
    _hint_count  = 0
    with _meta_engine.begin() as _conn:
        for _group in _hint_groups:
            _cid = _group.get("connection_id") or _name_to_uuid(_group.get("connection_name", ""))
            for _col in (_group.get("hints") or _group.get("columns") or []):
                _hint_key = f"{_group.get('connection_name', '')}__{_group.get('table_name', '')}__{_col['column_name']}"
                _insert_or_skip(_conn, _HINT_SQL, {
                    "hid": _name_to_uuid(_hint_key),
                    "cid": _cid,
                    "dfid": _group.get("dataflow_id"),
                    "sname": _group.get("schema_name"),
                    "tname": _group.get("table_name"),
                    "cname": _col["column_name"],
                    "dtype": _col["data_type"],
                    "fmt": _col.get("format"),
                    "prec": _col.get("precision"),
                    "scale": _col.get("scale"),
                    "defval": _col.get("default_value"),
                    "ordinal": _col.get("ordinal_position"),
                    "active": _col.get("is_active", True),
                    "now": _NOW,
                })
                _hint_count += 1
    print(f"Seeded {_hint_count} schema hint(s) across {len(_hint_groups)} group(s).")
    print("Setup complete.")

In [ ]:
# ---------------------------------------------------------------------------
# Run DataCoolie with DatabaseProvider
# ---------------------------------------------------------------------------
from datacoolie.engines.spark_engine import SparkEngine
from datacoolie.core.models import DataCoolieRunConfig
from datacoolie.metadata.database_provider import DatabaseProvider
from datacoolie.orchestration.driver import DataCoolieDriver
from datacoolie.platforms.fabric_platform import FabricPlatform

# Uncomment to use PolarsEngine instead of SparkEngine:
# from datacoolie.engines.polars_engine import PolarsEngine
# _token = notebookutils.credentials.getToken('storage')
# engine = PolarsEngine(
#     platform=FabricPlatform(),
#     storage_options={"bearer_token": _token, "use_fabric_endpoint": "true"},
# )
# run_config = DataCoolieRunConfig(max_workers=4)

try:
    spark_session = spark  # type: ignore[name-defined]
except NameError as _exc:
    raise RuntimeError(
        "This sample must run inside a Fabric notebook where `spark` is preloaded."
    ) from _exc

platform = FabricPlatform()
engine   = SparkEngine(spark_session, platform=platform)
run_config = DataCoolieRunConfig(max_workers=8)

# DatabaseProvider reads connections, dataflows, and schema hints directly
# from the Fabric SQL Database.  The pre-built _meta_engine is reused so
# no second connection string is required.
metadata = DatabaseProvider(engine=_meta_engine, workspace_id=WORKSPACE_ID)

with DataCoolieDriver(
    engine=engine,
    metadata_provider=metadata,
    base_log_path=BASE_LOG_PATH,
    config=run_config,
) as driver:
    result = driver.run(stage=STAGE)

print(
    "Result:",
    {
        "total":      result.total,
        "succeeded":  result.succeeded,
        "failed":     result.failed,
        "skipped":    result.skipped,
    },
)

In [ ]:
# ---------------------------------------------------------------------------
# Verify — row counts in the metadata DB
# ---------------------------------------------------------------------------
from sqlalchemy import text

_tables = [
    ("dc_framework_connections",  "workspace_id"),
    ("dc_framework_dataflows",    "workspace_id"),
    ("dc_framework_watermarks",   None),
    ("dc_framework_schema_hints", None),
]

with _meta_engine.connect() as _conn:
    print(f"Metadata DB: {SQL_DATABASE}  workspace_id={WORKSPACE_ID!r}")
    print("-" * 56)
    for _tbl, _ws_col in _tables:
        if _ws_col:
            _row = _conn.execute(
                text(f"SELECT COUNT(*) FROM {_tbl} WHERE {_ws_col} = :ws"),
                {"ws": WORKSPACE_ID},
            ).fetchone()
        else:
            _row = _conn.execute(text(f"SELECT COUNT(*) FROM {_tbl}")).fetchone()
        print(f"  {_tbl:<46}  {_row[0]:>5} rows")